# S3_NB3 — how much of the gap can a learned router capture?

**~5 GPU-hours · feature dump + a small gate per exit**

## The overclaim this fixes

Study 2 says the oracle excess "cannot be reached by any router". What was
actually shown is that **a second seed** cannot reach it. A learned router with
access to the input might do better, and nobody has measured it.

```
capture fraction = (router − confidence baseline) / (oracle_in − baseline)
```

**Pre-registered (H2):** a learned router captures **< 25 %** of the gap.

| outcome | reading |
|---|---|
| captures most | the field is right, the gap is real headroom, and here is a router |
| captures a little | the bound is mostly noise, now quantified |
| captures none | the strongest version of Study 2's claim |

All three are reportable and two are positive.

## The deployability constraint

A gate at exit *k* may use **only features available at exit k**. Anything else
is not a router, it is an oracle wearing a router's clothes — the exact mistake
`pred_depth` turned out to be in Study 2.

## The control that decides whether the number means anything

Train the gate on seed *i*, evaluate on seed *j*'s network. An in-seed capture
fraction alone is uninterpretable: a gate can fit one seed's noise perfectly.
**Both numbers are reported, always.**

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   6941a4b50582   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2cc4ba5e0935   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
    'dCBmb3IgdGhlIG9mZmxpbmUgY2hlY2tzIikK',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

In [ ]:
# === CELL 2 -- WHERE EVERYTHING LIVES ======================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine. Set MSC_ROOT explicitly if your runs are
# somewhere specific -- e.g. r'C:\msc_results'.

DATA_DIR = None      # e.g. r'E:\msc_data'      -- None = choose for me
MSC_ROOT = None      # e.g. r'C:\msc_results'   -- None = choose for me

# ---------------------------------------------------------------------------
import os
from pathlib import Path

M = msc                       # Study 2's notebooks say `M`; same module.

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_SCRATCH'] = MSC_ROOT

sess = M.Session(account='local', phase='p4', dataset='cifar100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print(f'msc_lib   {M.__version__}')
print(f'MSC_ROOT  {MSC_ROOT}')
print(f'data_dir  {sess.data_dir}')

# A Session CREATES runs/, so "the directory exists" proves nothing. Count the
# runs that actually carry a measurement -- that is what every cell below reads.
_runs_dir = Path(MSC_ROOT) / 'runs'
_measured = sorted(d.name for d in _runs_dir.iterdir()
                   if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists()
                   ) if _runs_dir.is_dir() else []
print(f'measured runs on disk: {len(_measured)}')
if not _measured:
    raise RuntimeError(
        f'no measured runs under {MSC_ROOT}/runs.\n'
        'Study 3 re-analyses Study 1 output. Fetch it first with\n'
        'notebooks_study2/S2_NB0_Fetch.ipynb, or set MSC_ROOT above\n'
        'to the folder that already holds them.')
print(f'  e.g. {_measured[0]}')

In [ ]:
# === Which runs am I analysing? ===========================================
# One accessor, so the dedupe rule and the emptiness checks live in a single
# place rather than being re-typed in each notebook (rule 4).
import numpy as np, pandas as pd

runs_dir = Path(MSC_ROOT) / 'runs'

def measured_runs(dataset='cifar100', methods=('base',), require=True):
    ids = sorted(d.name for d in runs_dir.iterdir()
                 if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists())
    if not ids:
        raise RuntimeError(f'no measured runs under {runs_dir}')
    df = pd.DataFrame([{**M.parse_run_id(r), 'run_id': r} for r in ids])

    for col in ('method', 'dataset', 'arch', 'seed', 'phase'):
        if col not in df.columns:
            raise RuntimeError(
                f'parse_run_id did not yield a {col!r} column. Columns present: '
                f'{list(df.columns)}. Run ids look like: {ids[:3]}')

    sel = df[(df['dataset'] == dataset) & (df['method'].isin(methods))]
    if require and sel.empty:
        raise RuntimeError('; '.join([
            f'{len(df)} measured run(s) on disk, but NONE with '
            f'dataset={dataset!r} and method in {tuple(methods)!r}',
            f'datasets present: {sorted(df["dataset"].dropna().unique())}',
            f'methods present: {sorted(df["method"].dropna().unique())}',
            'either the wrong MSC_ROOT is set, or the runs you need have not '
            'been fetched or trained yet']))

    # p0 pilots and p1 runs share seed numbers, so pooling them counts one seed
    # twice -- the contamination Study 2 found. Keep the highest phase.
    before = len(sel)
    sel = (sel.sort_values('phase')
              .drop_duplicates(subset=['arch', 'dataset', 'method', 'seed'],
                               keep='last'))
    if len(sel) < before:
        print(f'dropped {before - len(sel)} duplicate (arch, method, seed) '
              f'run(s) -- pilot replicates')
    return sel.reset_index(drop=True)

_b = measured_runs()
print(f'{len(_b)} CIFAR-100 base run(s); '
      f'{_b["arch"].nunique()} architecture(s)')

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

---
## Dump per-exit features

One forward pass per run. ~50 MB per run at 256 dims — cheap, and it is the
only thing standing between us and a learned router.

In [ ]:

import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path

ARCHS = ['resnet20', 'resnet32x4', 'vgg8']
SEEDS = [1, 2]          # two seeds: the cross-seed control needs them
feat_dir = Path(MSC_ROOT) / 'features'
feat_dir.mkdir(parents=True, exist_ok=True)

def dump_features(run_id, cfg):
    out = feat_dir / f'{run_id}.npz'
    if out.exists():
        return out
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    L = M.run_layout(sess.work, run_id)
    blob = torch.load(L['checkpoints'] / 'ckpt_best.pt', map_location=device,
                      weights_only=False)
    backbone = M.place_model(M.build_model(cfg['arch'], cfg['num_classes']),
                             device, cfg, tag='feature dump')
    backbone.load_state_dict(blob['model'], strict=True)
    me = M.place_model(M.MultiExitModel(backbone, cfg['num_classes'], freeze=True),
                       device, cfg)
    hp = M.exit_heads_path(sess.work, run_id)
    me.heads.load_state_dict(torch.load(hp, map_location=device,
                                        weights_only=False)['heads'])
    me.eval()

    _, val_loader, _, _, _ = M.build_loaders(cfg)
    feats, labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device, non_blocking=True)
            fs = me.backbone.forward_features(x)
            pooled = []
            for f in fs:
                if f.dim() == 4:
                    pooled.append(nn.functional.adaptive_avg_pool2d(f, 1)
                                  .flatten(1).float().cpu().numpy())
                elif f.dim() == 3:
                    pooled.append((f[:, 0] if me.token_model
                                   else f.mean(1)).float().cpu().numpy())
                else:
                    pooled.append(f.flatten(1).float().cpu().numpy())
            feats.append(pooled)
            labels.append(batch[1].numpy())
    K = len(feats[0])
    stacked = {f'f{k}': np.concatenate([b[k] for b in feats], axis=0)
               for k in range(K)}
    np.savez_compressed(out, label=np.concatenate(labels), **stacked)
    del me, backbone
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

paths = {}
for a in ARCHS:
    for s in SEEDS:
        cfg = M.base_config(a, 'cifar100', seed=s, phase='p1', method='base')
        rid = cfg['run_id']
        if not (Path(MSC_ROOT) / 'runs' / rid / 'per_sample' / 'test.parquet').exists():
            print(f'  missing {rid}, skipped')
            continue
        paths[rid] = dump_features(rid, cfg)
        mb = paths[rid].stat().st_size / 2**20
        print(f'  {rid:38s} {mb:6.1f} MB')
print(f'\n{len(paths)} feature dump(s)')

---
## Train a gate per exit, then measure capture

The gate is deliberately small. A large one would fit the seed's noise and
inflate the in-seed number — which the cross-seed control would then expose,
but it is cheaper not to invite the problem.

In [ ]:

from sklearn.linear_model import LogisticRegression   # small on purpose

def gate_scores(train_rid, eval_rid):
    '''Train per-exit gates on train_rid, score eval_rid's samples.'''
    tr = np.load(paths[train_rid]); ev = np.load(paths[eval_rid])
    dtr = pd.read_parquet(Path(MSC_ROOT) / 'runs' / train_rid
                          / 'per_sample' / 'test.parquet').sort_values('sample_idx')
    ks = sorted(int(c.split('_d')[1]) for c in dtr.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    lab_tr = dtr['label'].to_numpy()
    out = []
    for i, k in enumerate(ks[:-1]):        # no gate needed at the final exit
        y = (dtr[f'pred_d{k}'].to_numpy() == lab_tr).astype(int)
        Xtr, Xev = tr[f'f{i}'], ev[f'f{i}']
        if y.min() == y.max():
            out.append(np.full(len(Xev), float(y.mean())))
            continue
        clf = LogisticRegression(max_iter=300, C=0.1)
        clf.fit(Xtr, y)
        out.append(clf.predict_proba(Xev)[:, 1])
    return np.stack(out, axis=1), ks

def correctness(rid):
    d = pd.read_parquet(Path(MSC_ROOT) / 'runs' / rid / 'per_sample'
                        / 'test.parquet').sort_values('sample_idx')
    ks = sorted(int(c.split('_d')[1]) for c in d.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    lab = d['label'].to_numpy()
    corr = np.stack([(d[f'pred_d{k}'].to_numpy() == lab) for k in ks], axis=1).astype(float)
    conf = np.stack([d[f'top1p_d{k}'].to_numpy() for k in ks], axis=1)
    return corr, conf, ks

---
## Evaluate at matched budget

Routing helpers are **imported from Study 2's notebook logic**, re-implemented
here only because the notebook is standalone — but the canaries in
`tools/s2_routing_canaries.py` cover the same functions and must pass first.

In [ ]:

def _cost(k, rho):  return float(np.mean(np.asarray(rho)[k]))

def route_confidence(conf, correct, rho, target):
    n, K = correct.shape
    lo, hi = 0.0, 1.0
    for _ in range(60):
        th = (lo + hi) / 2
        fires = conf >= th; fires[:, -1] = True
        k = fires.argmax(axis=1); c = _cost(k, rho)
        if c < target: lo = th
        else: hi = th
    return float(correct[np.arange(n), k].mean()), c

def route_gate(pgate, correct, rho, target):
    '''Exit at the first exit whose gate probability clears a threshold.'''
    n, K = correct.shape
    lo, hi = 0.0, 1.0
    for _ in range(60):
        th = (lo + hi) / 2
        fires = np.concatenate([pgate >= th, np.ones((n, 1), bool)], axis=1)
        k = fires.argmax(axis=1); c = _cost(k, rho)
        if c < target: lo = th
        else: hi = th
    return float(correct[np.arange(n), k].mean()), c

def route_oracle(cc, ce, rho, target):
    rho = np.asarray(rho, float)
    lo, hi = 0.0, 100.0
    for _ in range(80):
        lam = (lo + hi) / 2
        k = (cc - lam * rho[None, :]).argmax(axis=1)
        if float(rho[k].mean()) > target: lo = lam
        else: hi = lam
    k = (cc - hi * rho[None, :]).argmax(axis=1)
    return float(ce[np.arange(len(ce)), k].mean()), float(rho[k].mean())

TARGET = 0.80
rows = []
for a in ARCHS:
    rids = [r for r in paths if M.parse_run_id(r)['arch'] == a]
    if len(rids) < 2:
        continue
    rho = M.load_or_build_budgets(a, sess.work, 'cifar100')['axes']['depth']['rho']
    i, j = sorted(rids)[0], sorted(rids)[1]
    for train_on, eval_on, kind in [(i, i, 'in-seed'), (i, j, 'cross-seed')]:
        corr, conf, ks = correctness(eval_on)
        base, _ = route_confidence(conf, corr, rho, TARGET)
        orac, _ = route_oracle(corr, corr, rho, TARGET)
        pg, _ = gate_scores(train_on, eval_on)
        gt, _ = route_gate(pg, corr, rho, TARGET)
        gap = orac - base
        rows.append({'arch': a, 'kind': kind, 'baseline': base * 100,
                     'router': gt * 100, 'oracle': orac * 100,
                     'gap': gap * 100, 'router_gain': (gt - base) * 100,
                     'capture': (gt - base) / gap if gap > 1e-9 else np.nan})

cap = pd.DataFrame(rows)
M.save_analysis(sess.data_dir, 's3_router_capture', cap)
print(cap.round(3).to_string(index=False))
print()
for kind in ['in-seed', 'cross-seed']:
    sub = cap[cap['kind'] == kind]
    if len(sub):
        print(f'  {kind:11s} median capture = {sub["capture"].median()*100:6.2f} %')
print()
cs = cap[cap['kind'] == 'cross-seed']['capture'].median()
print(f'H2 (< 25 % captured): '
      f'{"SUPPORTED" if cs < 0.25 else "FALSIFIED"}  (cross-seed {cs*100:.1f} %)')
print()
print('The CROSS-SEED number is the one that means anything. If in-seed capture')
print('is high and cross-seed is not, the gate memorised one seed s noise --')
print('which is Study 2 s finding restated, not a contradiction of it.')

---
## Canaries — the gate must be shown to work and to fail

In [ ]:

rng = np.random.default_rng(0)
n, K = 4000, 5
rho = [0.2, 0.4, 0.6, 0.8, 1.0]
easy = rng.random(n) < 0.5
cc = np.zeros((n, K)); cc[easy, :] = 1.0; cc[~easy, K-1] = 1.0

# a gate handed the truth must capture ~everything
perfect = np.repeat(easy[:, None].astype(float), K-1, axis=1)
b, _ = route_confidence(rng.random((n, K)), cc, rho, 0.7)
o, _ = route_oracle(cc, cc, rho, 0.7)
g, _ = route_gate(perfect, cc, rho, 0.7)
capture = (g - b) / (o - b) if o > b else float('nan')
print(f'{"PASS" if capture > 0.8 else "FAIL"}  oracle-gate captures '
      f'{capture*100:.0f}% (must be ~100)')

# a gate handed noise must capture ~nothing
g2, _ = route_gate(rng.random((n, K-1)), cc, rho, 0.7)
cap2 = (g2 - b) / (o - b) if o > b else float('nan')
print(f'{"PASS" if cap2 < 0.25 else "FAIL"}  noise-gate captures '
      f'{cap2*100:.0f}% (must be ~0)')

---
## Next

Record in `study3/03_LOG.md`. Then `S3_NB4_Pruning` (Q3), which is independent
of everything above.